In [1]:
!pip install cfgrib
!apt-get install -y libeccodes-dev

Defaulting to user installation because normal site-packages is not writeable
/bin/bash: apt-get: command not found


In [2]:
import gzip
import shutil

gz_path = "MRMS_MESH_Max_60min_00.50_20240731-210000.grib2.gz"
grib_path = "MRMS_MESH_Max_60min_00.50_20240731-210000.grib2"

with gzip.open(gz_path, "rb") as f_in:
    with open(grib_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

#Citation: ChatGPT

In [3]:
import xarray as xr

ds_mesh = xr.open_dataset(grib_path, engine="cfgrib")

ds_mesh

Ignoring index file 'MRMS_MESH_Max_60min_00.50_20240731-210000.grib2.923a8.idx' older than GRIB file


<xarray.Dataset>
Dimensions:         (latitude: 3500, longitude: 7000)
Coordinates:
    time            datetime64[ns] ...
    step            timedelta64[ns] ...
    heightAboveSea  float64 ...
  * latitude        (latitude) float64 54.99 54.98 54.98 ... 20.03 20.02 20.01
  * longitude       (longitude) float64 230.0 230.0 230.0 ... 300.0 300.0 300.0
    valid_time      datetime64[ns] ...
Data variables:
    unknown         (latitude, longitude) float32 ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             161
    GRIB_centreDescription:  161
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             161
    history:                 2026-04-16T20:47 GRIB to CDM+CF via cfgrib-0.9.1...

In [4]:
ds_mesh = ds_mesh.rename({'unknown':'mesh'})
ds_mesh

<xarray.Dataset>
Dimensions:         (latitude: 3500, longitude: 7000)
Coordinates:
    time            datetime64[ns] ...
    step            timedelta64[ns] ...
    heightAboveSea  float64 ...
  * latitude        (latitude) float64 54.99 54.98 54.98 ... 20.03 20.02 20.01
  * longitude       (longitude) float64 230.0 230.0 230.0 ... 300.0 300.0 300.0
    valid_time      datetime64[ns] ...
Data variables:
    mesh            (latitude, longitude) float32 ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             161
    GRIB_centreDescription:  161
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             161
    history:                 2026-04-16T20:47 GRIB to CDM+CF via cfgrib-0.9.1...

In [5]:
print(ds_mesh.time.values)

2024-07-31T21:00:00.000000000


In [6]:
!pip install herbie-data

Defaulting to user installation because normal site-packages is not writeable


In [7]:
from herbie import Herbie

In [8]:
H = Herbie("2024-07-31 21:00", model="hrrr", product="sfc", fxx=0)

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2024-Jul-31 21:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws


In [9]:
search = [
    "CAPE:surface",
    "CAPE:90-0 mb above ground",
    "CIN:surface",
    "MXUPHL:5000-2000 m above ground",
    "HLCY:3000-0 m above ground",
    "HGT:level of adiabatic condensation from sfc",
    "REFC:entire atmosphere",
    "TMP:500 mb",
    "TMP:2 m above ground",
    "DPT:2 m above ground",
    "UGRD:10 m above ground",
    "VGRD:10 m above ground",
    "UGRD:700 mb",
    "VGRD:700 mb"
]

date = ['2024-07-31 21:00']

all_ds = []

for i in date:
    H = Herbie(i, model="hrrr", product="sfc", fxx=0)

    var_ds = []
    for s in search:
        files = H.download(search=s)
        ds = xr.open_mfdataset(
            files,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""}
        )
        var_ds.append(ds)

    merged = xr.merge(var_ds, compat='override')
    all_ds.append(merged)

ds_hrrr = xr.concat(all_ds, dim='time')

#SOURCE: ChatGPT

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2024-Jul-31 21:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws


In [10]:
ds_hrrr = ds_hrrr.rename({'unknown':'mxuphl'})

In [11]:
ds_hrrr

<xarray.Dataset>
Dimensions:                  (time: 1, y: 1059, x: 1799)
Coordinates:
  * time                     (time) datetime64[ns] 2024-07-31T21:00:00
    step                     timedelta64[ns] ...
    surface                  float64 ...
    latitude                 (y, x) float64 dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    longitude                (y, x) float64 dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    valid_time               datetime64[ns] ...
    pressureFromGroundLayer  float64 ...
    heightAboveGroundLayer   float64 ...
    adiabaticCondensation    float64 ...
    atmosphere               float64 ...
    isobaricInhPa            float64 ...
    heightAboveGround        float64 ...
Dimensions without coordinates: y, x
Data variables: (12/13)
    cape                     (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    cin                      (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    mxuphl                   (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    hlcy                     (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    gh                       (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    refc                     (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    ...                       ...
    t2m                      (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    d2m                      (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    u10                      (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    v10                      (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    u                        (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    v                        (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-16T20:48 GRIB to CDM+CF via cfgrib-0.9.1...

In [12]:
print(ds_hrrr.time.values)

['2024-07-31T21:00:00.000000000']


In [13]:
mesh_on_hrrr = ds_mesh.interp(
    latitude=ds_hrrr.latitude,
    longitude=ds_hrrr.longitude,
    method="linear")

ds_combined = xr.merge([ds_hrrr, mesh_on_hrrr])

#SOURCE: ChatGPT

In [14]:
ds_combined

<xarray.Dataset>
Dimensions:                  (time: 1, y: 1059, x: 1799)
Coordinates: (12/13)
  * time                     (time) datetime64[ns] 2024-07-31T21:00:00
    step                     timedelta64[ns] 00:00:00
    surface                  float64 0.0
    latitude                 (y, x) float64 dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    longitude                (y, x) float64 dask.array<chunksize=(1059, 1799), meta=np.ndarray>
    valid_time               datetime64[ns] 2024-07-31T21:00:00
    ...                       ...
    heightAboveGroundLayer   float64 5e+03
    adiabaticCondensation    float64 0.0
    atmosphere               float64 0.0
    isobaricInhPa            float64 500.0
    heightAboveGround        float64 2.0
    heightAboveSea           float64 500.0
Dimensions without coordinates: y, x
Data variables: (12/14)
    cape                     (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    cin                      (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    mxuphl                   (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    hlcy                     (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    gh                       (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    refc                     (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    ...                       ...
    d2m                      (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    u10                      (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    v10                      (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    u                        (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    v                        (time, y, x) float32 dask.array<chunksize=(1, 1059, 1799), meta=np.ndarray>
    mesh                     (y, x) float64 -3.0 -3.0 -3.0 ... -1.0 -1.0 -1.0
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-04-16T20:48 GRIB to CDM+CF via cfgrib-0.9.1...

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
import pandas as pd

In [16]:
time_flat = ds_combined['time'].values.flatten()
lat_flat = ds_combined['latitude'].values.flatten()
lon_flat = ds_combined['longitude'].values.flatten()
mesh_flat = ds_combined['mesh'].values.flatten()
cape_flat = ds_combined['cape'].values.flatten()
cin_flat = ds_combined['cin'].values.flatten()
mxuphl_flat = ds_combined['mxuphl'].values.flatten()
hlcy_flat = ds_combined['hlcy'].values.flatten()
gh_flat = ds_combined['gh'].values.flatten()
refc_flat = ds_combined['refc'].values.flatten()
t500_flat = ds_combined['t'].values.flatten()
t2m_flat = ds_combined['t2m'].values.flatten()
d2m_flat = ds_combined['d2m'].values.flatten()
u10_flat = ds_combined['u10'].values.flatten()
v10_flat = ds_combined['v10'].values.flatten()
u_flat = ds_combined['u'].values.flatten()
v_flat = ds_combined['v'].values.flatten()
  #I'm sure there's an less pepega way of doing this, but this is the way my pea brain understands :D

In [17]:
cape_x = preprocessing.StandardScaler().fit_transform(cape_flat.reshape(-1,1))
cin_x = preprocessing.StandardScaler().fit_transform(cin_flat.reshape(-1,1))
mxuphl_x = preprocessing.StandardScaler().fit_transform(mxuphl_flat.reshape(-1,1))
hlcy_x = preprocessing.StandardScaler().fit_transform(hlcy_flat.reshape(-1,1))
gh_x = preprocessing.StandardScaler().fit_transform(gh_flat.reshape(-1,1))
refc_x = preprocessing.StandardScaler().fit_transform(refc_flat.reshape(-1,1))
t500_x = preprocessing.StandardScaler().fit_transform(t500_flat.reshape(-1,1))
t2m_x = preprocessing.StandardScaler().fit_transform(t2m_flat.reshape(-1,1))
d2m_x = preprocessing.StandardScaler().fit_transform(d2m_flat.reshape(-1,1))
u10_x = preprocessing.StandardScaler().fit_transform(u10_flat.reshape(-1,1))
v10_x = preprocessing.StandardScaler().fit_transform(v10_flat.reshape(-1,1))
u_x = preprocessing.StandardScaler().fit_transform(u_flat.reshape(-1,1))
v_x = preprocessing.StandardScaler().fit_transform(v_flat.reshape(-1,1))
  #standardize predictors (don't need to fit MESH since that is what we're looking to predict)

In [23]:
df_original = pd.DataFrame({'latitude': lat_flat,
                   'longitude': lon_flat,
                   'MESH': mesh_flat,
                   'CAPE': cape_x.flatten(),
                   'CIN': cin_x.flatten(),
                   'MXUPHL': mxuphl_x.flatten(),
                   'HLCY': hlcy_x.flatten(),
                   'GH': gh_x.flatten(),
                   'REFC': refc_x.flatten(),
                   'T500': t500_x.flatten(),
                   'T2M': t2m_x.flatten(),
                   'D2M': d2m_x.flatten(),
                   'U10': u10_x.flatten(),
                   'V10': v10_x.flatten(),
                   'U': u_x.flatten(),
                   'V': v_x.flatten()})

In [25]:
df = df_original.dropna()
df = df[(df['latitude'] >= 43.50) & (df['latitude'] <= 49.38) & (df['longitude'] >= (360-97.24)) & (df['longitude'] <= (360-89.49))]
df

,latitude,longitude,MESH,CAPE,CIN,MXUPHL,HLCY,GH,REFC,T500,T2M,D2M,U10,V10,U,V
1287192,43.508441,262.789996,-1.0,3.090155,0.298569,0.0,-0.040376,-0.778904,-0.254089,-1.962466,0.303885,1.172544,-1.131777,1.287244,1.352731,2.085943
1287193,43.508351,262.827048,-1.0,3.145105,0.298569,0.0,0.122293,-0.845293,-0.254089,-1.962466,0.179300,1.203341,-1.068991,1.205585,1.378967,2.070701
1287194,43.508250,262.864100,-1.0,3.186317,0.298569,0.0,0.075816,-0.901649,-0.254089,-1.962466,0.054715,1.218740,-0.880631,1.156590,1.405203,2.070701
1287195,43.508139,262.901152,-1.0,3.158842,0.298569,0.0,-0.063614,-0.905594,-0.254089,-1.928085,0.023568,1.195642,-0.734129,1.140258,1.431439,2.055459
1287196,43.508016,262.938204,-1.0,3.076418,0.298569,0.0,-0.203044,-0.877980,-0.254089,-1.893705,0.065097,1.172544,-0.608556,1.172922,1.470793,2.040217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1697549,49.375809,270.347733,-1.0,1.441661,0.104202,0.0,1.632786,-0.374036,-0.254089,-2.271891,-0.059489,0.541195,0.207669,0.029694,1.588855,-0.520471
1697550,49.373549,270.388233,-1.0,1.482874,0.104202,0.0,1.772216,-0.461164,-0.254089,-2.271891,-0.090635,0.556594,0.123953,-0.035633,1.575737,-0.520471
1697551,49.371278,270.428730,-1.0,1.482874,-0.090164,0.0,1.888408,-0.508842,-0.254089,-2.271891,-0.173692,0.595091,0.061167,-0.100960,1.562619,-0.520471
1697552,49.368995,270.469224,-1.0,1.427924,-0.090164,0.0,1.911646,-0.509857,-0.254089,-2.271891,-0.184074,0.579692,0.040238,-0.133624,1.536383,-0.520471


In [26]:
y = df['MESH']
X = df.drop(columns=['MESH'])
print(len(X), len(y))

44061 44061


In [27]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size = 0.1)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, train_size=0.8)

In [28]:
from sklearn.tree import plot_tree
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(max_depth=20, min_samples_leaf = 25, n_estimators = 200, random_state = 100).fit(X=X_train, y=y_train)
rf

RandomForestRegressor(max_depth=20, min_samples_leaf=25, n_estimators=200,
                      random_state=100)

In [29]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred = rf.predict(X_test)

r2 = r2_score(y_test, y_pred)
mean_sq_er = mean_squared_error(y_test, y_pred)
mean_abs_er = mean_absolute_error(y_test, y_pred)

print(f"Test R2 Score: is {r2:.2f}")
print(f"Test Mean Squared Error is {mean_sq_er:.2f}")
print(f"Test Mean Absolute Error is {mean_abs_er:.2f}")

Test R2 Score: is 0.52
Test Mean Squared Error is 1.08
Test Mean Absolute Error is 0.24


In [31]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

In [32]:
disp = ConfusionMatrixDisplay.from_estimator(
        rf,
        X_test,
        y_test,
        cmap=plt.cm.Blues,
        normalize=None,
    )
disp.ax_.set_title("Confusion Matrix")

plt.show()

# # Plot base confusion matrix
# base_matrix = metrics.confusion_matrix(expected_base, predicted_base)
# base_matrix_plot = metrics.ConfusionMatrixDisplay(confusion_matrix = base_matrix, display_labels = ['MD', 'Watch']).plot(ax=ax[0], cmap = plt.cm.Blues, 
#                                                                                                                          colorbar = False) 
# base_matrix_plot.im_.set_clim(vmin = 0, vmax = 40)


# # Plot optimized confusion matrix                 
# opt_matrix = metrics.confusion_matrix(expected_opt, predicted_opt)
# opt_matrix_plot = metrics.ConfusionMatrixDisplay(confusion_matrix = opt_matrix, display_labels = ['MD', 'Watch']).plot(ax = ax[1], cmap = plt.cm.Reds, 
#                                                                                                                        colorbar = False)
# opt_matrix_plot.im_.set_clim(vmin = 0, vmax = 40)

# plt.tight_layout()
# plt.savefig('figure_10.jpg')

ValueError: ConfusionMatrixDisplay.from_estimator only supports classifiers